E-Commerce Customer & Sales Analytics
Python (pandas, NumPy, Matplotlib)
RFM segmentation and sales trend analysis



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import os

warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)

In [ ]:
# ── Style ──────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f8f8f8",
    "axes.grid":        True,
    "grid.color":       "white",
    "grid.linewidth":   1.2,
    "font.family":      "DejaVu Sans",
    "axes.spines.top":  False,
    "axes.spines.right":False,
})
PALETTE = ["#2C3E7A", "#E05A2B", "#3AAA6E", "#F0A500", "#8E44AD"]

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. GENERATE REALISTIC DATASET
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("  E-COMMERCE CUSTOMER & SALES ANALYTICS")
print("=" * 60)
print("\n[1/5] Generating dataset...")

np.random.seed(42)
N_ORDERS   = 98_000
N_CUSTOMERS = 85_000

categories = {
    "Electronics":        0.18,
    "Clothing & Fashion": 0.15,
    "Health & Beauty":    0.14,
    "Sports & Outdoors":  0.11,
    "Home & Garden":      0.10,
    "Books & Media":      0.09,
    "Toys & Games":       0.08,
    "Food & Grocery":     0.08,
    "Automotive":         0.04,
    "Office Supplies":    0.03,
}
states = ["SP","RJ","MG","RS","PR","SC","BA","GO","ES","PE"]
state_weights = [0.30,0.15,0.12,0.09,0.08,0.06,0.05,0.04,0.03,0.08]

start = datetime(2022, 1, 1)
end   = datetime(2024, 6, 30)
date_range = (end - start).days

# Simulate seasonal purchase patterns
day_offsets = np.random.choice(
    date_range,
    size=N_ORDERS,
    p=np.tile(
        np.concatenate([
            np.linspace(0.8, 1.4, 91),   # Q1 ramp
            np.linspace(1.4, 0.9, 91),   # Q2 decline
            np.linspace(0.9, 1.2, 91),   # Q3 mid
            np.linspace(1.2, 1.8, 92),   # Q4 peak (Black Friday / Christmas)
        ]),
        (date_range // 365 + 2)
    )[:date_range] / (
        np.tile(
            np.concatenate([
                np.linspace(0.8, 1.4, 91),
                np.linspace(1.4, 0.9, 91),
                np.linspace(0.9, 1.2, 91),
                np.linspace(1.2, 1.8, 92),
            ]),
            (date_range // 365 + 2)
        )[:date_range].sum()
    )
)

order_dates    = [start + timedelta(days=int(d)) for d in day_offsets]
customer_ids   = np.random.randint(1, N_CUSTOMERS + 1, N_ORDERS)
cat_names      = list(categories.keys())
cat_probs      = list(categories.values())
order_cats     = np.random.choice(cat_names, size=N_ORDERS, p=cat_probs)

# Price distributions per category
price_params = {
    "Electronics":        (320, 180),
    "Clothing & Fashion": (85,  45),
    "Health & Beauty":    (55,  30),
    "Sports & Outdoors":  (130, 70),
    "Home & Garden":      (110, 60),
    "Books & Media":      (35,  18),
    "Toys & Games":       (60,  35),
    "Food & Grocery":     (40,  20),
    "Automotive":         (200, 120),
    "Office Supplies":    (50,  28),
}
prices = np.array([
    max(9.9, np.random.normal(*price_params[c]))
    for c in order_cats
])

statuses = np.random.choice(
    ["delivered","delivered","delivered","delivered","shipped","cancelled","refunded"],
    size=N_ORDERS
)
delivery_days = np.where(
    statuses == "delivered",
    np.random.gamma(shape=3, scale=4, size=N_ORDERS).astype(int) + 1,
    np.nan
)
review_scores = np.where(
    statuses == "delivered",
    np.random.choice([1,2,3,4,5], size=N_ORDERS, p=[0.04,0.07,0.14,0.35,0.40]),
    np.nan
)
states_col = np.random.choice(states, size=N_ORDERS, p=state_weights)

df = pd.DataFrame({
    "order_id":       range(1, N_ORDERS + 1),
    "customer_id":    customer_ids,
    "order_date":     order_dates,
    "category":       order_cats,
    "price":          prices.round(2),
    "status":         statuses,
    "delivery_days":  delivery_days,
    "review_score":   review_scores,
    "state":          states_col,
})
df["order_date"] = pd.to_datetime(df["order_date"])
df["year_month"]  = df["order_date"].dt.to_period("M")
df["year"]        = df["order_date"].dt.year
df["month"]       = df["order_date"].dt.month
df["quarter"]     = df["order_date"].dt.quarter

df.to_csv("orders.csv", index=False)
print(f"   ✓ {len(df):,} orders · {df['customer_id'].nunique():,} unique customers · {df['category'].nunique()} categories")


  E-COMMERCE CUSTOMER & SALES ANALYTICS

[1/5] Generating dataset...
   ✓ 98,000 orders · 58,110 unique customers · 10 categories


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. DATA QUALITY CHECK
# ══════════════════════════════════════════════════════════════════════════════
print("\n[2/5] Data quality check...")

print(f"   Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0].to_string()}")
print(f"   Date range : {df['order_date'].min().date()} → {df['order_date'].max().date()}")
print(f"   Price range: €{df['price'].min():.2f} – €{df['price'].max():.2f}  |  Median: €{df['price'].median():.2f}")
print(f"   Status dist:\n{df['status'].value_counts(normalize=True).mul(100).round(1).to_string()}")



[2/5] Data quality check...
   Missing values:
delivery_days    41809
review_score     41809
   Date range : 2022-01-01 → 2024-06-29
   Price range: €9.90 – €969.95  |  Median: €77.85
   Status dist:
status
delivered    57.3
shipped      14.3
refunded     14.3
cancelled    14.1


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. SALES TREND ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("\n[3/5] Sales trend analysis...")

delivered = df[df["status"] == "delivered"].copy()
monthly   = delivered.groupby("year_month").agg(
    revenue=("price", "sum"),
    orders=("order_id", "count"),
    aov=("price", "mean")
).reset_index()
monthly["year_month_dt"] = monthly["year_month"].dt.to_timestamp()
monthly["revenue_12m_avg"] = monthly["revenue"].rolling(3, min_periods=1).mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Sales Performance Overview", fontsize=15, fontweight="bold", y=1.01)

# Revenue trend
ax = axes[0, 0]
ax.fill_between(monthly["year_month_dt"], monthly["revenue"] / 1000, alpha=0.18, color=PALETTE[0])
ax.plot(monthly["year_month_dt"], monthly["revenue"] / 1000, color=PALETTE[0], linewidth=2)
ax.plot(monthly["year_month_dt"], monthly["revenue_12m_avg"] / 1000, color=PALETTE[1],
        linewidth=1.5, linestyle="--", label="3-month avg")
ax.set_title("Monthly Revenue (€ thousands)", fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{x:.0f}k"))
ax.legend(fontsize=9)

# Revenue by category
ax = axes[0, 1]
cat_rev = delivered.groupby("category")["price"].sum().sort_values(ascending=True)
bars = ax.barh(cat_rev.index, cat_rev.values / 1000, color=PALETTE[0], edgecolor="white")
ax.set_title("Total Revenue by Category (€k)", fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{x:.0f}k"))
for bar, val in zip(bars, cat_rev.values):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f"€{val/1000:.0f}k", va="center", fontsize=8)

# Monthly order volume
ax = axes[1, 0]
ax.bar(monthly["year_month_dt"], monthly["orders"], color=PALETTE[2], width=20, edgecolor="white")
ax.set_title("Monthly Order Volume", fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# AOV over time
ax = axes[1, 1]
ax.plot(monthly["year_month_dt"], monthly["aov"], color=PALETTE[3], linewidth=2, marker="o", markersize=3)
ax.set_title("Average Order Value (€)", fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{x:.0f}"))

plt.tight_layout()
plt.savefig("outputs/01_sales_trends.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✓ Saved: outputs/01_sales_trends.png")



[3/5] Sales trend analysis...
   ✓ Saved: outputs/01_sales_trends.png


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. RFM CUSTOMER SEGMENTATION
# ══════════════════════════════════════════════════════════════════════════════
print("\n[4/5] RFM customer segmentation...")

snapshot_date = delivered["order_date"].max() + timedelta(days=1)

rfm = delivered.groupby("customer_id").agg(
    recency=("order_date",  lambda x: (snapshot_date - x.max()).days),
    frequency=("order_id",  "count"),
    monetary=("price",      "sum")
).reset_index()

# Score 1–5 using rank-based percentile (robust to duplicates)
def score_col(series, ascending=True):
    ranked = series.rank(method="first", ascending=ascending)
    return pd.cut(ranked, bins=5, labels=[1,2,3,4,5]).astype(int)

rfm["recency_score"]   = score_col(rfm["recency"],   ascending=False)
rfm["frequency_score"] = score_col(rfm["frequency"],  ascending=True)
rfm["monetary_score"]  = score_col(rfm["monetary"],   ascending=True)

rfm["rfm_score"]  = rfm["recency_score"] + rfm["frequency_score"] + rfm["monetary_score"]
rfm["rfm_class"]  = rfm["rfm_score"].apply(lambda s:
    "Champions"        if s >= 13 else
    "Loyal Customers"  if s >= 11 else
    "Potential Loyal"  if s >= 9  else
    "At Risk"          if s >= 7  else
    "Lost"
)

seg_summary = rfm.groupby("rfm_class").agg(
    customers=("customer_id", "count"),
    avg_revenue=("monetary", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_recency=("recency", "mean")
).round(1)

print(f"\n   RFM Segment Breakdown:")
print(seg_summary.to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("RFM Customer Segmentation", fontsize=15, fontweight="bold")

order_segs = ["Champions","Loyal Customers","Potential Loyal","At Risk","Lost"]
colors_seg = [PALETTE[2], PALETTE[0], PALETTE[3], PALETTE[1], "#AAAAAA"]

# Segment size
ax = axes[0]
seg_counts = rfm["rfm_class"].value_counts().reindex(order_segs)
ax.barh(seg_counts.index, seg_counts.values, color=colors_seg, edgecolor="white")
ax.set_title("Customers per Segment", fontweight="bold")
for i, v in enumerate(seg_counts.values):
    ax.text(v + 50, i, f"{v:,}", va="center", fontsize=9)

# Avg revenue per segment
ax = axes[1]
seg_rev = rfm.groupby("rfm_class")["monetary"].mean().reindex(order_segs)
ax.barh(seg_rev.index, seg_rev.values, color=colors_seg, edgecolor="white")
ax.set_title("Avg Revenue per Customer (€)", fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"€{x:.0f}"))

# Recency vs Monetary scatter
ax = axes[2]
for seg, col in zip(order_segs, colors_seg):
    sub = rfm[rfm["rfm_class"] == seg]
    ax.scatter(sub["recency"], sub["monetary"], alpha=0.3, s=8, color=col, label=seg)
ax.set_xlabel("Recency (days since last order)")
ax.set_ylabel("Total Revenue (€)")
ax.set_title("Recency vs Revenue", fontweight="bold")
ax.legend(fontsize=7, markerscale=2)

plt.tight_layout()
plt.savefig("outputs/02_rfm_segmentation.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✓ Saved: outputs/02_rfm_segmentation.png")


[4/5] RFM customer segmentation...

   RFM Segment Breakdown:
                 customers  avg_revenue  avg_frequency  avg_recency
rfm_class                                                          
At Risk              10005        109.4            1.0        458.1
Champions             6195        371.5            2.3        150.5
Lost                  9206         53.7            1.0        638.8
Loyal Customers       6872        240.0            1.7        286.6
Potential Loyal       8768        168.1            1.2        355.4
   ✓ Saved: outputs/02_rfm_segmentation.png


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# 5. DELIVERY & SATISFACTION ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("\n[5/5] Delivery & satisfaction analysis...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Delivery Performance & Customer Satisfaction", fontsize=15, fontweight="bold")

# Delivery time distribution
ax = axes[0]
del_data = delivered["delivery_days"].dropna()
ax.hist(del_data, bins=30, color=PALETTE[0], edgecolor="white", alpha=0.85)
ax.axvline(del_data.median(), color=PALETTE[1], linestyle="--", linewidth=2,
           label=f"Median: {del_data.median():.0f}d")
ax.set_title("Delivery Time Distribution", fontweight="bold")
ax.set_xlabel("Days to Deliver")
ax.legend()

# Review score distribution
ax = axes[1]
score_counts = delivered["review_score"].dropna().astype(int).value_counts().sort_index()
bars = ax.bar(score_counts.index, score_counts.values,
              color=[PALETTE[1],PALETTE[1],PALETTE[3],PALETTE[2],PALETTE[2]],
              edgecolor="white")
ax.set_title("Review Score Distribution", fontweight="bold")
ax.set_xlabel("Review Score (1–5)")
for bar, val in zip(bars, score_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{val:,}", ha="center", fontsize=9)

# Avg review score by delivery speed bucket
ax = axes[2]
delivered2 = delivered.copy()
delivered2["speed_bucket"] = pd.cut(
    delivered2["delivery_days"],
    bins=[0, 3, 7, 14, 21, 999],
    labels=["1–3d","4–7d","8–14d","15–21d","21d+"]
)
speed_score = delivered2.groupby("speed_bucket", observed=True)["review_score"].mean()
bars = ax.bar(speed_score.index, speed_score.values, color=PALETTE[0], edgecolor="white")
ax.set_ylim(1, 5.2)
ax.set_title("Avg Review Score by Delivery Speed", fontweight="bold")
ax.set_xlabel("Delivery Window")
ax.set_ylabel("Avg Score")
for bar, val in zip(bars, speed_score.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("outputs/03_delivery_satisfaction.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✓ Saved: outputs/03_delivery_satisfaction.png")



[5/5] Delivery & satisfaction analysis...
   ✓ Saved: outputs/03_delivery_satisfaction.png


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  KEY FINDINGS")
print("=" * 60)
print(f"  Total revenue analysed : €{delivered['price'].sum():,.0f}")
print(f"  Delivered orders       : {len(delivered):,} ({len(delivered)/len(df)*100:.1f}% of total)")
print(f"  Unique customers       : {rfm.shape[0]:,}")
print(f"  Champions (top segment): {(rfm['rfm_class']=='Champions').sum():,} customers")
print(f"  Avg delivery time      : {delivered['delivery_days'].mean():.1f} days")
print(f"  Avg review score       : {delivered['review_score'].mean():.2f} / 5.0")
print(f"\n  Output charts saved to ./outputs/")
print("=" * 60)


  KEY FINDINGS
  Total revenue analysed : €7,012,991
  Delivered orders       : 56,191 (57.3% of total)
  Unique customers       : 41,046
  Champions (top segment): 6,195 customers
  Avg delivery time      : 12.5 days
  Avg review score       : 4.00 / 5.0

  Output charts saved to ./outputs/
